# NFL Game-Line Model: Full Version

This is the finished NFL model. It replaces the baseline notebook and does everything end to end:

1. **Data**: every game since 1999 with closing lines, plus every play (for efficiency stats)
2. **Ratings**: Elo, opponent-adjusted passing and rushing efficiency (EPA per play), and quarterback ratings
3. **Models**: three candidates trained walk-forward; the winner is chosen on 2010–2015 only
4. **Market calibration**: learns how much of the model's disagreement with the line to believe
5. **Final exam**: forecast accuracy and a betting backtest on 2016 onward, seasons the model choice never saw
6. **This week's games**: probabilities, edges and each pick's historical track record
7. **Saved files**: the model, team ratings, track record and picks, ready for the website's backend

**How to use it:** click **Runtime → Run all**. The first run downloads every season of play-by-play and takes a minute or two. After that it reuses saved files and only refreshes the current season.

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import joblib
from scipy.stats import norm
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


# ---- Odds math (from the first notebook) ----
def american_to_decimal(odds):
    odds = np.asarray(odds, dtype=float)
    size = np.abs(odds)
    return np.where(odds < 0, 1 + 100 / size, 1 + size / 100)


def implied_prob(odds):
    odds = np.asarray(odds, dtype=float)
    size = np.abs(odds)
    return np.where(odds < 0, size / (size + 100), 100 / (size + 100))


def remove_vig(odds_a, odds_b):
    pa, pb = implied_prob(odds_a), implied_prob(odds_b)
    total = pa + pb
    return pa / total, pb / total, total - 1


def expected_value(p, odds):
    b = american_to_decimal(odds) - 1
    return np.asarray(p, dtype=float) * b - (1 - np.asarray(p, dtype=float))

---
# 1. Data

## 1a. Games and closing lines

Same source as before. This time the table keeps **upcoming games that already have a line**, so the model can price them at the end.

In [ ]:
GAMES_URLS = [
    "https://github.com/nflverse/nfldata/raw/master/data/games.csv",
    "https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv",
]
RAW_CACHE = "nflverse_games_raw.csv"
TEAM_FIX = {"OAK": "LV", "SD": "LAC", "STL": "LA"}   # relocated franchises keep one name


def load_games():
    # Always try for the newest file (it changes every week); fall back to the saved copy.
    for url in GAMES_URLS:
        try:
            df = pd.read_csv(url)
            df.to_csv(RAW_CACHE, index=False)
            return df
        except Exception:
            continue
    if os.path.exists(RAW_CACHE):
        print("Could not reach nflverse; using the saved copy of the games file.")
        return pd.read_csv(RAW_CACHE)
    raise RuntimeError("Could not download the games file and no saved copy exists.")


raw = load_games()
games = raw.copy()
games["gameday"] = pd.to_datetime(games["gameday"])
games["home_team"] = games["home_team"].replace(TEAM_FIX)
games["away_team"] = games["away_team"].replace(TEAM_FIX)
games["played"] = games["result"].notna()

# Keep finished games with a closing spread, and upcoming games that already have a line.
games = games[games["spread_line"].notna()].sort_values(["gameday", "game_id"]).reset_index(drop=True)

CURRENT_SEASON = int(games["season"].max())
season_done = not raw.loc[raw["season"] == CURRENT_SEASON, "result"].isna().any()
LAST_FULL_SEASON = CURRENT_SEASON if season_done else CURRENT_SEASON - 1

games["neutral"] = (games["location"] == "Neutral").astype(int)
games["home_field"] = 1 - games["neutral"]
games["home_win"] = np.where(games["result"] == 0, np.nan, (games["result"] > 0).astype(float))
games.loc[~games["played"], "home_win"] = np.nan

has_ml = games["home_moneyline"].notna() & games["away_moneyline"].notna()
games.loc[has_ml, "fair_home_prob"] = remove_vig(
    games.loc[has_ml, "home_moneyline"], games.loc[has_ml, "away_moneyline"])[0]

print(f"Finished games: {games['played'].sum():,} (seasons {games['season'].min()}–{CURRENT_SEASON})")
print(f"Upcoming games with a line: {(~games['played']).sum()}")
print(f"Last full season: {LAST_FULL_SEASON}   Current season: {CURRENT_SEASON}")

## 1b. Play-by-play efficiency

Final scores are noisy: a tipped pass or a fumble return can swing a game by 14 points without saying much about which team is better. **EPA (expected points added) per play** measures how much each play improved a team's scoring chances, so it describes how well a team actually moved the ball. It predicts future results better than past scores do.

This cell downloads every play since 1999 and boils it down to two small tables:

- **team_game**: each team's passing and rushing EPA in each game
- **qb_game**: each quarterback's EPA per dropback in each game

Both are saved, so only the current season is downloaded again on later runs.

In [ ]:
PBP_URL = "https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{season}.parquet"
PBP_COLS = ["game_id", "posteam", "defteam", "epa", "success", "pass", "rush",
            "qb_dropback", "qb_epa", "passer_player_id", "two_point_attempt"]
TEAM_CACHE, QB_CACHE = "nfl_team_game_stats.csv", "nfl_qb_game_stats.csv"


def summarize_season(pbp, season):
    # Turns one season of plays into one row per team per game, and one row per passer per game.
    plays = pbp[((pbp["pass"] == 1) | (pbp["rush"] == 1)) & pbp["epa"].notna()
                & pbp["posteam"].notna() & pbp["defteam"].notna()
                & (pbp["posteam"] != "") & (pbp["defteam"] != "")
                & (pbp["two_point_attempt"].fillna(0) != 1)].copy()
    plays["is_pass"] = (plays["pass"] == 1).astype(int)
    plays["pass_epa"] = plays["epa"].where(plays["is_pass"] == 1, 0.0)
    plays["rush_epa"] = plays["epa"].where(plays["is_pass"] == 0, 0.0)
    team = plays.groupby(["game_id", "posteam", "defteam"], as_index=False).agg(
        plays=("epa", "size"), pass_plays=("is_pass", "sum"),
        pass_epa=("pass_epa", "sum"), rush_epa=("rush_epa", "sum"), success=("success", "mean"))
    team["rush_plays"] = team["plays"] - team["pass_plays"]
    team = team.rename(columns={"posteam": "team", "defteam": "opp"})
    team["team"], team["opp"] = team["team"].replace(TEAM_FIX), team["opp"].replace(TEAM_FIX)

    drops = pbp[(pbp["qb_dropback"] == 1) & pbp["qb_epa"].notna() & pbp["passer_player_id"].notna()]
    qb = drops.groupby(["game_id", "posteam", "passer_player_id"], as_index=False).agg(
        dropbacks=("qb_epa", "size"), qb_epa=("qb_epa", "sum"))
    qb = qb.rename(columns={"posteam": "team", "passer_player_id": "qb_id"})
    qb["team"] = qb["team"].replace(TEAM_FIX)
    team.insert(0, "season", season); qb.insert(0, "season", season)
    return team, qb


def load_efficiency(first_season, last_season):
    team_cache = pd.read_csv(TEAM_CACHE) if os.path.exists(TEAM_CACHE) else pd.DataFrame()
    qb_cache = pd.read_csv(QB_CACHE) if os.path.exists(QB_CACHE) else pd.DataFrame()
    have = set(team_cache["season"].unique()) if len(team_cache) else set()
    teams, qbs, started = [], [], time.time()
    for season in range(first_season, last_season + 1):
        cached = season in have
        if cached and season != last_season:      # finished seasons never change
            teams.append(team_cache[team_cache["season"] == season])
            qbs.append(qb_cache[qb_cache["season"] == season])
            continue
        try:
            pbp = pd.read_parquet(PBP_URL.format(season=season), columns=PBP_COLS)
            t, q = summarize_season(pbp, season)
            print(f"  {season}: {len(pbp):,} plays -> {len(t)} team-games   ({time.time() - started:.0f}s elapsed)")
        except Exception as e:
            if cached:
                print(f"  {season}: could not refresh, using the saved copy")
                t, q = team_cache[team_cache["season"] == season], qb_cache[qb_cache["season"] == season]
            else:
                print(f"  {season}: not available ({type(e).__name__}); skipping")
                continue
        teams.append(t); qbs.append(q)
    team_game, qb_game = pd.concat(teams, ignore_index=True), pd.concat(qbs, ignore_index=True)
    team_game.to_csv(TEAM_CACHE, index=False); qb_game.to_csv(QB_CACHE, index=False)
    return team_game, qb_game


print("Loading play-by-play efficiency (the first run downloads every season)...")
team_game, qb_game = load_efficiency(int(games["season"].min()), CURRENT_SEASON)

check = games[games["played"]].merge(
    team_game[["game_id", "team", "plays"]], left_on=["game_id", "home_team"], right_on=["game_id", "team"], how="left")
print(f"\n{len(team_game):,} team-games and {len(qb_game):,} quarterback-games")
print(f"Finished games matched to play-by-play: {check['plays'].notna().mean():.1%}")
print(f"League average: {team_game['pass_epa'].sum() / team_game['pass_plays'].sum():+.3f} EPA per pass play, "
      f"{team_game['rush_epa'].sum() / team_game['rush_plays'].sum():+.3f} per rush, "
      f"{team_game['plays'].mean():.0f} plays per team per game")

---
# 2. Ratings and features

One function walks through every game in date order and keeps four kinds of running ratings. For each game it **first saves the pre-game numbers, then updates with the result**, exactly like Elo in the baseline notebook. That ordering is what keeps future information out.

| Rating | What it measures | How it updates |
|---|---|---|
| **Elo** | Overall team strength from wins and margins | Winner takes points from the loser |
| **Efficiency** (4 per team) | Pass offense, rush offense, pass defense, rush defense, in EPA per play | Moves toward each game's EPA, **adjusted for the opponent**: a good day against a great defense counts for more |
| **Quarterback** | Each QB's EPA per dropback | Recent games count more; new QBs start below average until they have a track record |
| **Scoring form and pace** | Recent points for/against and plays per game | Weighted average favoring recent games |

The quarterback rating matters most when a starter is out. Team ratings still reflect the usual starter, so the model also gets **how the starting QB compares to that team's usual QB**.

In [ ]:
# Settings are fixed in advance and are not tuned on the test seasons.
ELO_K, ELO_HFA, ELO_REVERT, ELO_START = 20.0, 55.0, 1 / 3, 1500.0
EFF_ALPHA = 0.10          # weight of the newest game in the efficiency ratings
EFF_REVERT = 1 / 3        # pull toward league average between seasons
LEAGUE_ALPHA = 0.002      # the league-average baseline moves slowly
FORM_ALPHA, LEAGUE_PTS, LEAGUE_PLAYS = 0.10, 22.0, 63.0
QB_DECAY = 0.96           # each new game shrinks the weight of older ones
QB_PRIOR_DROPBACKS = 150  # an unknown QB is treated like 150 dropbacks of below-average play
QB_PRIOR_PENALTY = 0.10   # ...this far below the league-average EPA per dropback
USUAL_QB_ALPHA = 0.15

team_stats = {(g, t): (pp, pe, rp, re, n) for g, t, pp, pe, rp, re, n in zip(
    team_game["game_id"], team_game["team"], team_game["pass_plays"], team_game["pass_epa"],
    team_game["rush_plays"], team_game["rush_epa"], team_game["plays"])}
qb_stats = {(g, q): (d, e) for g, q, d, e in zip(
    qb_game["game_id"], qb_game["qb_id"], qb_game["dropbacks"], qb_game["qb_epa"])}


def build_features(df):
    elo, last_season, last_qb = {}, {}, {}
    eff = {}                                   # team -> [off_pass, off_rush, def_pass, def_rush]
    form = {}                                  # team -> [points_for, points_against, plays]
    qb = {}                                    # qb_id -> [weighted_epa, weighted_dropbacks]
    usual_qb = {}                              # team -> rating of the QBs it has been starting
    league = {"pass": 0.0, "rush": -0.05, "qb": 0.0}
    rows = []

    def qb_value(qb_id):
        prior_mean = league["qb"] - QB_PRIOR_PENALTY
        s, n = qb.get(qb_id, (0.0, 0.0))
        return (s + prior_mean * QB_PRIOR_DROPBACKS) / (n + QB_PRIOR_DROPBACKS)

    for g in df.itertuples(index=False):
        home, away = g.home_team, g.away_team
        for team in (home, away):
            if team not in elo:
                elo[team], eff[team], form[team] = ELO_START, [0.0] * 4, [LEAGUE_PTS, LEAGUE_PTS, LEAGUE_PLAYS]
            elif last_season[team] < g.season:      # new season: pull everything toward average
                elo[team] += ELO_REVERT * (ELO_START - elo[team])
                eff[team] = [x * (1 - EFF_REVERT) for x in eff[team]]
                form[team] = [form[team][0] + 0.5 * (LEAGUE_PTS - form[team][0]),
                              form[team][1] + 0.5 * (LEAGUE_PTS - form[team][1]),
                              form[team][2] + 0.5 * (LEAGUE_PLAYS - form[team][2])]
            last_season[team] = g.season

        # Starting quarterbacks. For upcoming games with no listed starter, assume the last one.
        home_qb = g.home_qb_id if isinstance(g.home_qb_id, str) else last_qb.get(home)
        away_qb = g.away_qb_id if isinstance(g.away_qb_id, str) else last_qb.get(away)
        hq, aq = qb_value(home_qb), qb_value(away_qb)
        h_usual, a_usual = usual_qb.get(home, hq), usual_qb.get(away, aq)

        # ---- 1) Save everything known BEFORE kickoff ----
        rows.append({
            "home_elo": elo[home], "away_elo": elo[away],
            "home_off_pass": eff[home][0], "home_off_rush": eff[home][1], "home_def_pass": eff[home][2], "home_def_rush": eff[home][3],
            "away_off_pass": eff[away][0], "away_off_rush": eff[away][1], "away_def_pass": eff[away][2], "away_def_rush": eff[away][3],
            "home_pf": form[home][0], "home_pa": form[home][1], "home_pace": form[home][2],
            "away_pf": form[away][0], "away_pa": form[away][1], "away_pace": form[away][2],
            "home_qb_value": hq, "away_qb_value": aq,
            "home_qb_vs_usual": hq - h_usual, "away_qb_vs_usual": aq - a_usual,
            "home_qb_used": home_qb, "away_qb_used": away_qb,
        })
        if not g.played:
            continue

        # ---- 2) Only now use the result to update the ratings ----
        gap = elo[home] - elo[away] + (0 if g.neutral else ELO_HFA)
        expected_home = 1 / (1 + 10 ** (-gap / 400))
        actual_home = 1.0 if g.result > 0 else (0.0 if g.result < 0 else 0.5)
        winner_gap = gap if g.result > 0 else (-gap if g.result < 0 else 0.0)
        blowout = np.log(max(abs(g.result), 1) + 1) * 2.2 / (winner_gap * 0.001 + 2.2)
        change = ELO_K * blowout * (actual_home - expected_home)
        elo[home] += change; elo[away] -= change

        pre = {home: list(eff[home]), away: list(eff[away])}
        for team, opp in ((home, away), (away, home)):
            stats = team_stats.get((g.game_id, team))
            if stats is None:
                continue
            pass_plays, pass_epa, rush_plays, rush_epa, n_plays = stats
            for kind, plays, epa, off_i, def_i in (("pass", pass_plays, pass_epa, 0, 2), ("rush", rush_plays, rush_epa, 1, 3)):
                if plays < 5:
                    continue
                seen = epa / plays - league[kind]                       # vs. league average
                eff[team][off_i] += EFF_ALPHA * ((seen - pre[opp][def_i]) - pre[team][off_i])
                eff[opp][def_i] += EFF_ALPHA * ((seen - pre[team][off_i]) - pre[opp][def_i])
                league[kind] += LEAGUE_ALPHA * (epa / plays - league[kind])
            form[team][2] += FORM_ALPHA * (n_plays - form[team][2])

        form[home][0] += FORM_ALPHA * (g.home_score - form[home][0]); form[home][1] += FORM_ALPHA * (g.away_score - form[home][1])
        form[away][0] += FORM_ALPHA * (g.away_score - form[away][0]); form[away][1] += FORM_ALPHA * (g.home_score - form[away][1])

        for team, starter, value in ((home, home_qb, hq), (away, away_qb, aq)):
            if starter is None:
                continue
            last_qb[team] = starter
            usual_qb[team] = usual_qb.get(team, value) + USUAL_QB_ALPHA * (value - usual_qb.get(team, value))
            stats = qb_stats.get((g.game_id, starter))
            if stats is not None:
                dropbacks, epa = stats
                s, n = qb.get(starter, (0.0, 0.0))
                qb[starter] = (s * QB_DECAY + epa, n * QB_DECAY + dropbacks)
                league["qb"] += LEAGUE_ALPHA * 5 * (epa / dropbacks - league["qb"])

    state = {"elo": elo, "eff": eff, "form": form, "qb_value": qb_value, "last_qb": last_qb, "league": league}
    return pd.DataFrame(rows, index=df.index), state


features, state = build_features(games)
games = games.join(features)
print(f"Built pre-game ratings for {len(games):,} games.")
print(f"League baselines now: pass {state['league']['pass']:+.3f}, rush {state['league']['rush']:+.3f}, QB {state['league']['qb']:+.3f} EPA per play")

### Do the ratings look right?

A face-validity check before any modeling: the current top and bottom teams, and the highest-rated active quarterbacks. If these lists look like the real league, the ratings are working.

In [ ]:
qb_names = pd.concat([
    games[["home_qb_id", "home_qb_name"]].set_axis(["id", "name"], axis=1),
    games[["away_qb_id", "away_qb_name"]].set_axis(["id", "name"], axis=1)]).dropna().drop_duplicates("id", keep="last").set_index("id")["name"]

teams_now = sorted(state["elo"])
ratings = pd.DataFrame({
    "elo": [state["elo"][t] for t in teams_now],
    "pass_offense": [state["eff"][t][0] for t in teams_now],
    "rush_offense": [state["eff"][t][1] for t in teams_now],
    "pass_defense": [-state["eff"][t][2] for t in teams_now],     # flipped so higher = better defense
    "rush_defense": [-state["eff"][t][3] for t in teams_now],
    "points_for": [state["form"][t][0] for t in teams_now],
    "points_against": [state["form"][t][1] for t in teams_now],
    "quarterback": [qb_names.get(state["last_qb"].get(t), "unknown") for t in teams_now],
    "qb_rating": [state["qb_value"](state["last_qb"].get(t)) for t in teams_now],
}, index=pd.Index(teams_now, name="team")).sort_values("elo", ascending=False)

print("TOP 8 AND BOTTOM 4 TEAMS RIGHT NOW (efficiency in EPA per play vs. league average; higher is better)")
display(pd.concat([ratings.head(8), ratings.tail(4)]).round({"elo": 0, "pass_offense": 3, "rush_offense": 3, "pass_defense": 3,
                                                           "rush_defense": 3, "points_for": 1, "points_against": 1, "qb_rating": 3}))
print("HIGHEST-RATED CURRENT STARTING QUARTERBACKS")
display(ratings.sort_values("qb_rating", ascending=False)[["quarterback", "qb_rating"]].head(8).round(3))
ratings.round(4).to_csv("nfl_team_ratings.csv")

## Turning ratings into model inputs

For the **margin**, what matters is the gap between the two teams, so most inputs are home minus away. For the **total**, what matters is how much scoring both teams produce together, so those inputs are sums.

Weather is filled in conservatively: indoor games get zero wind, and outdoor games with no reading get a typical value. Upcoming games have no weather yet, so they use the typical values too.

In [ ]:
games["elo_diff"] = games["home_elo"] - games["away_elo"]
games["rest_diff"] = (games["home_rest"].fillna(7) - games["away_rest"].fillna(7)).clip(-7, 7)
for part in ("off_pass", "off_rush", "def_pass", "def_rush"):
    games[f"{part}_diff"] = games[f"home_{part}"] - games[f"away_{part}"]
    games[f"{part}_sum"] = games[f"home_{part}"] + games[f"away_{part}"]
games["qb_diff"] = games["home_qb_value"] - games["away_qb_value"]
games["qb_sum"] = games["home_qb_value"] + games["away_qb_value"]
games["qb_vs_usual_diff"] = games["home_qb_vs_usual"] - games["away_qb_vs_usual"]
games["qb_vs_usual_sum"] = games["home_qb_vs_usual"] + games["away_qb_vs_usual"]
games["form_diff"] = (games["home_pf"] - games["home_pa"]) - (games["away_pf"] - games["away_pa"])
games["pace_sum"] = games["home_pace"] + games["away_pace"]

# Roof: if a retractable-roof game has no listing yet, assume that stadium's usual setting.
indoor_known = games["roof"].isin(["dome", "closed"]).astype(float).where(games["roof"].notna())
usual_indoor = indoor_known.groupby(games["home_team"]).transform("mean").fillna(0).round()
games["indoor"] = indoor_known.fillna(usual_indoor).astype(int)
games["wind_filled"] = np.where(games["indoor"] == 1, 0.0, games["wind"].fillna(8.0))
games["cold"] = np.where(games["indoor"] == 1, 0.0, (40 - games["temp"].fillna(60)).clip(lower=0))

MARGIN_BASE = ["elo_diff", "home_field", "rest_diff", "div_game"]
MARGIN_FULL = MARGIN_BASE + ["off_pass_diff", "off_rush_diff", "def_pass_diff", "def_rush_diff",
                             "qb_diff", "qb_vs_usual_diff", "form_diff"]
TOTAL_BASE = ["home_pf", "home_pa", "away_pf", "away_pa", "indoor", "wind_filled"]
TOTAL_FULL = TOTAL_BASE + ["off_pass_sum", "off_rush_sum", "def_pass_sum", "def_rush_sum",
                           "qb_sum", "qb_vs_usual_sum", "pace_sum", "cold"]

missing = games[MARGIN_FULL + TOTAL_FULL].isna().sum()
assert missing.sum() == 0, f"Missing feature values: {missing[missing > 0].to_dict()}"
print(f"Margin model inputs: {len(MARGIN_FULL)}   Total model inputs: {len(TOTAL_FULL)}   No missing values.")

---
# 3. Models

Three candidates, all trained walk-forward (each season is predicted by a model that has only seen earlier seasons):

| Candidate | Inputs | Method |
|---|---|---|
| **Baseline** | Elo, home field, rest, division | Ridge regression (the previous notebook's model) |
| **Full ridge** | Everything above plus efficiency, quarterbacks and form | Ridge regression |
| **Boosted trees** | Same as full ridge | Gradient boosting, which can find interactions a straight-line model can't |
| **Blend** | — | Average of full ridge and boosted trees |

## The honest way to choose

Picking the model that looks best on *all* the data and then reporting that same score is cheating: the choice itself was fitted to the test. So the seasons are split by job:

- **2002 onward**: training (always only seasons before the one being predicted)
- **2010–2015**: *validation*. Used to choose the model.
- **2016 onward**: *holdout*. Looked at once, after every choice is locked in. This is the number to believe.

In [ ]:
FIRST_TRAIN_SEASON = 2002        # 1999–2001 only warm up the ratings
FIRST_TEST_SEASON = 2006
VALID_SEASONS = (2010, 2015)
HOLDOUT_START = 2016


def ridge():
    return make_pipeline(StandardScaler(), Ridge(alpha=10.0))


def boosted():
    return HistGradientBoostingRegressor(max_depth=3, learning_rate=0.04, max_iter=250,
                                         min_samples_leaf=40, l2_regularization=1.0, random_state=0)


def walk_forward(df, feature_cols, target, make_model):
    # Predict each season with a model trained only on finished games from earlier seasons.
    pred = pd.Series(np.nan, index=df.index)
    for season in range(FIRST_TEST_SEASON, CURRENT_SEASON + 1):
        train = df[df["played"] & (df["season"] >= FIRST_TRAIN_SEASON) & (df["season"] < season)]
        test = df[df["season"] == season]
        if len(test) == 0:
            continue
        model = make_model().fit(train[feature_cols], train[target])
        pred.loc[test.index] = model.predict(test[feature_cols])
    return pred


started = time.time()
margin_preds = pd.DataFrame({
    "Baseline": walk_forward(games, MARGIN_BASE, "result", ridge),
    "Full ridge": walk_forward(games, MARGIN_FULL, "result", ridge),
    "Boosted trees": walk_forward(games, MARGIN_FULL, "result", boosted),
})
margin_preds["Blend"] = (margin_preds["Full ridge"] + margin_preds["Boosted trees"]) / 2
total_preds = pd.DataFrame({
    "Baseline": walk_forward(games, TOTAL_BASE, "total", ridge),
    "Full ridge": walk_forward(games, TOTAL_FULL, "total", ridge),
    "Boosted trees": walk_forward(games, TOTAL_FULL, "total", boosted),
})
total_preds["Blend"] = (total_preds["Full ridge"] + total_preds["Boosted trees"]) / 2
n_seasons = CURRENT_SEASON - FIRST_TEST_SEASON + 1
print(f"Walk-forward done: {n_seasons} seasons x 3 model types x 2 targets = {n_seasons * 6} models trained in {time.time() - started:.0f}s")

## 3a. Choosing the model on validation seasons only

Lower RMSE (typical miss in points) is better. The market's own number is shown for scale. The candidate with the lowest validation RMSE becomes the final model; the holdout column is **not** printed here on purpose.

In [ ]:
def rmse(error):
    return float(np.sqrt((error ** 2).mean()))


valid = games["played"] & games["season"].between(*VALID_SEASONS)
holdout = games["played"] & (games["season"] >= HOLDOUT_START)

margin_valid = {name: rmse(games.loc[valid, "result"] - margin_preds.loc[valid, name]) for name in margin_preds}
total_valid = {name: rmse(games.loc[valid, "total"] - total_preds.loc[valid, name]) for name in total_preds}
MARGIN_CHOICE = min(margin_valid, key=margin_valid.get)
TOTAL_CHOICE = min(total_valid, key=total_valid.get)

selection = pd.DataFrame({"margin_rmse": margin_valid, "total_rmse": total_valid}).round(3)
selection.loc["Closing line (the market)"] = [
    round(rmse(games.loc[valid, "result"] - games.loc[valid, "spread_line"]), 3),
    round(rmse(games.loc[valid, "total"] - games.loc[valid, "total_line"]), 3)]
print(f"Validation seasons {VALID_SEASONS[0]}–{VALID_SEASONS[1]} ({int(valid.sum()):,} games)")
display(selection)
print(f"Chosen margin model: {MARGIN_CHOICE}")
print(f"Chosen total model:  {TOTAL_CHOICE}")

games["pred_margin"] = margin_preds[MARGIN_CHOICE]
games["pred_total"] = total_preds[TOTAL_CHOICE]
games["base_margin"] = margin_preds["Baseline"]
games["base_total"] = total_preds["Baseline"]

## 3b. What does the model lean on?

These are the full ridge model's weights, trained on every finished season. Inputs are put on the same scale first, so the sizes are comparable: each number is how many points the prediction moves when that input rises by one typical step.

In [ ]:
train_all = games[games["played"] & (games["season"] >= FIRST_TRAIN_SEASON)]
margin_ridge = ridge().fit(train_all[MARGIN_FULL], train_all["result"])
total_ridge = ridge().fit(train_all[TOTAL_FULL], train_all["total"])

weights_margin = pd.Series(margin_ridge[-1].coef_, index=MARGIN_FULL).sort_values(key=abs, ascending=False)
weights_total = pd.Series(total_ridge[-1].coef_, index=TOTAL_FULL).sort_values(key=abs, ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, w, title in ((axes[0], weights_margin, "Margin model"), (axes[1], weights_total, "Total model")):
    ax.barh(w.index[::-1], w.values[::-1], color=np.where(w.values[::-1] >= 0, "tab:blue", "tab:red"))
    ax.axvline(0, color="k", linewidth=1)
    ax.set_title(f"{title}: points per typical step of each input")
plt.tight_layout(); plt.show()
hfa = (margin_ridge.predict(train_all[MARGIN_FULL].assign(home_field=1)).mean()
       - margin_ridge.predict(train_all[MARGIN_FULL].assign(home_field=0)).mean())
print(f"Home-field advantage the margin model learned: {hfa:+.2f} points")

---
# 4. Market calibration

The baseline notebook found that the model's disagreements with the spread were mostly its own mistakes: only a small fraction of each point of disagreement showed up in results. A raw model that says "home by 9" against a line of "home by 3" should not be shown to anyone as a 67% cover probability.

So the final step learns **how much of the disagreement to believe**:

`final margin = spread + c × (model margin − spread)`

`c` is estimated from past out-of-sample predictions only, and re-estimated every season:

- `c = 0` means "trust the line completely" (no bets, ever)
- `c = 1` means "trust the model completely"
- a small `c` means the model's opinion nudges the line a little

The same is done for totals. Moneyline probabilities start from the vig-free moneyline and are nudged by the same amount, so they stay anchored to the real price.

**This is the number the website should show.** The raw probability is kept alongside it for comparison.

In [ ]:
FIRST_CALIBRATED_SEASON = 2010     # needs a few seasons of out-of-sample predictions first


def believed_share(actual, line, model):
    # Share of the model-vs-line disagreement that shows up in results (0 to 1).
    gap = model - line
    c = float(((actual - line) * gap).sum() / (gap ** 2).sum())
    return min(max(c, 0.0), 1.0)


for col in ("c_margin", "c_total", "sigma_margin", "sigma_total", "sigma_margin_raw", "sigma_total_raw"):
    games[col] = np.nan
history = []
for season in range(FIRST_CALIBRATED_SEASON, CURRENT_SEASON + 1):
    past = games[games["played"] & games["season"].between(FIRST_TEST_SEASON, season - 1)]
    past_t = past.dropna(subset=["total_line"])
    c_m = believed_share(past["result"], past["spread_line"], past["pred_margin"])
    c_t = believed_share(past_t["total"], past_t["total_line"], past_t["pred_total"])
    rows = games["season"] == season
    games.loc[rows, "c_margin"], games.loc[rows, "c_total"] = c_m, c_t
    games.loc[rows, "sigma_margin"] = (past["result"] - (past["spread_line"] + c_m * (past["pred_margin"] - past["spread_line"]))).std()
    games.loc[rows, "sigma_total"] = (past_t["total"] - (past_t["total_line"] + c_t * (past_t["pred_total"] - past_t["total_line"]))).std()
    games.loc[rows, "sigma_margin_raw"] = (past["result"] - past["pred_margin"]).std()
    games.loc[rows, "sigma_total_raw"] = (past_t["total"] - past_t["pred_total"]).std()
    history.append({"season": season, "c_margin": c_m, "c_total": c_t})
history = pd.DataFrame(history).set_index("season")

games["final_margin"] = games["spread_line"] + games["c_margin"] * (games["pred_margin"] - games["spread_line"])
games["final_total"] = games["total_line"] + games["c_total"] * (games["pred_total"] - games["total_line"])

# Probabilities. "raw" trusts the model fully; the others are market-calibrated.
games["p_cover_raw"] = norm.cdf((games["pred_margin"] - games["spread_line"]) / games["sigma_margin_raw"])
games["p_over_raw"] = norm.cdf((games["pred_total"] - games["total_line"]) / games["sigma_total_raw"])
games["p_cover"] = norm.cdf((games["final_margin"] - games["spread_line"]) / games["sigma_margin"])
games["p_over"] = norm.cdf((games["final_total"] - games["total_line"]) / games["sigma_total"])
shift = games["c_margin"] * (games["pred_margin"] - games["spread_line"]) / games["sigma_margin"]
anchor = norm.ppf(games["fair_home_prob"].clip(0.005, 0.995))
games["p_win"] = np.where(games["fair_home_prob"].notna(), norm.cdf(anchor + shift),
                          norm.cdf(games["final_margin"] / games["sigma_margin"]))
games["p_win_raw"] = norm.cdf(games["pred_margin"] / games["sigma_margin_raw"])

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(history.index, history["c_margin"], marker="o", label="Spreads")
ax.plot(history.index, history["c_total"], marker="o", label="Totals")
ax.set_ylim(0, max(0.5, history.max().max() * 1.2)); ax.set_xlabel("Season being predicted")
ax.set_ylabel("Share of the disagreement believed (c)")
ax.set_title("How much the model's disagreement with the line is worth")
ax.legend(); plt.show()
latest = history.iloc[-1]
print(f"For {CURRENT_SEASON}: c = {latest['c_margin']:.2f} for spreads and {latest['c_total']:.2f} for totals.")
print(f"So when the model disagrees with a spread by 5 points, the final margin moves {5 * latest['c_margin']:.1f} points off the line.")

---
# 5. Final exam: holdout seasons

Everything is now locked: the features, the model choice, the calibration method and the betting rules below. This section opens the holdout seasons (2016 onward) for the first time.

## 5a. Forecast accuracy

In [ ]:
h = games[holdout].copy()
h_ml = h.dropna(subset=["fair_home_prob", "home_win"])
h_tot = h.dropna(subset=["total_line"])


def brier(p, won):
    return float(((p - won) ** 2).mean())


scorecard = pd.DataFrame({
    "margin_rmse": {
        "Baseline model (previous notebook)": rmse(h["result"] - h["base_margin"]),
        "Final model, raw": rmse(h["result"] - h["pred_margin"]),
        "Final model, market-calibrated": rmse(h["result"] - h["final_margin"]),
        "Closing line (the market)": rmse(h["result"] - h["spread_line"]),
    },
    "total_rmse": {
        "Baseline model (previous notebook)": rmse(h_tot["total"] - h_tot["base_total"]),
        "Final model, raw": rmse(h_tot["total"] - h_tot["pred_total"]),
        "Final model, market-calibrated": rmse(h_tot["total"] - h_tot["final_total"]),
        "Closing line (the market)": rmse(h_tot["total"] - h_tot["total_line"]),
    },
    "win_prob_brier": {
        "Baseline model (previous notebook)": brier(norm.cdf(h_ml["base_margin"] / h_ml["sigma_margin_raw"]), h_ml["home_win"]),
        "Final model, raw": brier(h_ml["p_win_raw"], h_ml["home_win"]),
        "Final model, market-calibrated": brier(h_ml["p_win"], h_ml["home_win"]),
        "Closing line (the market)": brier(h_ml["fair_home_prob"], h_ml["home_win"]),
    },
})
print(f"Holdout: {len(h):,} games, seasons {HOLDOUT_START}–{int(h['season'].max())}. Lower is better in every column.")
print(f"Final model = {MARGIN_CHOICE} for margin, {TOTAL_CHOICE} for total.")
display(scorecard.round({"margin_rmse": 2, "total_rmse": 2, "win_prob_brier": 4}))

gain = scorecard["margin_rmse"].iloc[0] - scorecard["margin_rmse"].iloc[1]
gap_before = scorecard["margin_rmse"].iloc[0] - scorecard["margin_rmse"].iloc[3]
print(f"Efficiency and quarterback ratings cut the typical margin miss by {gain:.2f} points, "
      f"closing {gain / gap_before:.0%} of the baseline's gap to the market.")

## 5b. Does the model know anything the market doesn't?

The same test as the baseline notebook, on holdout seasons: predict the result from the closing spread **and** the model's margin together. The weight on the model is the share of its disagreement that is real.

In [ ]:
def information_test(df, actual, line, model, label):
    fit = sm.OLS(df[actual], sm.add_constant(df[[line, model]])).fit()
    lo, hi = fit.conf_int().loc[model]
    return {"model": label, "weight_on_model": fit.params[model], "95%_low": lo, "95%_high": hi,
            "p_value": fit.pvalues[model], "weight_on_line": fit.params[line]}


info = pd.DataFrame([
    information_test(h, "result", "spread_line", "base_margin", "Spread: baseline model"),
    information_test(h, "result", "spread_line", "pred_margin", "Spread: final model"),
    information_test(h_tot, "total", "total_line", "base_total", "Total: baseline model"),
    information_test(h_tot, "total", "total_line", "pred_total", "Total: final model"),
]).set_index("model")
display(info.round(3))

c_now = float(info.loc["Spread: final model", "weight_on_model"])
worth = norm.pdf(0) * c_now * 4 / float(h["sigma_margin"].mean())
print(f"Spreads: a 4-point disagreement is worth about {c_now * 4:+.1f} real points, or {worth * 100:+.1f} percentage points of cover probability.")
print("Break-even at -110 needs +2.4 points of probability.")

## 5c. Betting backtest

Two strategies, both with rules set before looking at the holdout seasons. Stakes are 1 unit, pushes are refunded, and prices are the closing prices in the data.

| Strategy | Rule | What it tests |
|---|---|---|
| **Raw model** | Bet when the raw model probability beats break-even by 3 points or more | What happens if you believe the model outright |
| **Calibrated** | Bet when the market-calibrated probability beats break-even by 0.5 points or more | What the website would actually recommend |

Reminder from the baseline notebook: closing lines are the hardest prices to beat, and the recorded prices here are cheaper than the −110 most US sportsbooks charge. `roi_at_-110_%` is the fairer column for a typical bettor.

In [ ]:
RAW_MIN_EDGE = 0.03
CAL_MIN_EDGE = 0.005


def side_rows(df, market, side, p_raw, p_cal, price, other_price, won, pick):
    out = pd.DataFrame({
        "game_id": df["game_id"], "season": df["season"], "gameday": df["gameday"], "played": df["played"],
        "matchup": df["away_team"] + " at " + df["home_team"],
        "market": market, "side": side, "pick": pick,
        "model_prob_raw": p_raw, "model_prob": p_cal, "price": price, "other_price": other_price, "won": won})
    return out.dropna(subset=["model_prob", "price", "other_price"])


def build_candidates(df):
    # Every possible bet (both sides of spread, total and moneyline) for the given games.
    res, tot = df["result"], df["total"]
    home_cover = np.where(res.isna() | (res == df["spread_line"]), np.nan, (res > df["spread_line"]).astype(float))
    over = np.where(tot.isna() | df["total_line"].isna() | (tot == df["total_line"]), np.nan, (tot > df["total_line"]).astype(float))
    sh, sa = df["home_spread_odds"].fillna(-110), df["away_spread_odds"].fillna(-110)
    ov, un = df["over_odds"].fillna(-110), df["under_odds"].fillna(-110)
    line = df["spread_line"]
    fmt = lambda s, x: s + " " + x.map(lambda v: "PK" if v == 0 else f"{v:+g}")
    c = pd.concat([
        side_rows(df, "spread", "home", df["p_cover_raw"], df["p_cover"], sh, sa, home_cover, fmt(df["home_team"], -line)),
        side_rows(df, "spread", "away", 1 - df["p_cover_raw"], 1 - df["p_cover"], sa, sh, 1 - home_cover, fmt(df["away_team"], line)),
        side_rows(df, "total", "over", df["p_over_raw"], df["p_over"], ov, un, over, "Over " + df["total_line"].map("{:g}".format)),
        side_rows(df, "total", "under", 1 - df["p_over_raw"], 1 - df["p_over"], un, ov, 1 - over, "Under " + df["total_line"].map("{:g}".format)),
        side_rows(df, "moneyline", "home", df["p_win_raw"], df["p_win"], df["home_moneyline"], df["away_moneyline"], df["home_win"], df["home_team"] + " to win"),
        side_rows(df, "moneyline", "away", 1 - df["p_win_raw"], 1 - df["p_win"], df["away_moneyline"], df["home_moneyline"], 1 - df["home_win"], df["away_team"] + " to win"),
    ], ignore_index=True)
    c["break_even"] = implied_prob(c["price"])
    c["market_prob"] = remove_vig(c["price"], c["other_price"])[0]
    c["edge_raw"] = c["model_prob_raw"] - c["break_even"]
    c["edge"] = c["model_prob"] - c["break_even"]
    c["ev"] = expected_value(c["model_prob"], c["price"])
    c["profit"] = np.where(c["won"].isna(), 0.0, np.where(c["won"] == 1, american_to_decimal(c["price"]) - 1, -1.0))
    return c


def roi_range(b, n_boot=2000, seed=0):
    # 95% range for ROI, by resampling the bets many times.
    profit = b["profit"].to_numpy()
    boot = profit[np.random.default_rng(seed).integers(0, len(profit), size=(n_boot, len(profit)))].mean(axis=1)
    return float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


def summarize(b):
    decided = b["won"].notna()
    n = int(decided.sum())
    if n == 0:
        return pd.Series({"bets": 0})
    hit = float(b.loc[decided, "won"].mean())
    half = 1.96 * np.sqrt(hit * (1 - hit) / n)
    profit = b["profit"].to_numpy()
    low, high = roi_range(b)
    at_110 = (hit * (100 / 110) - (1 - hit)) * n / len(b)
    return pd.Series({
        "bets": n, "hit_rate_%": round(hit * 100, 1),
        "hit_95%_range": f"{(hit - half) * 100:.1f}–{(hit + half) * 100:.1f}",
        "break_even_%": round(float(b.loc[decided, "break_even"].mean()) * 100, 1),
        "roi_%": round(float(profit.mean()) * 100, 1),
        "roi_95%_range": f"{low * 100:+.1f} to {high * 100:+.1f}",
        "roi_at_-110_%": "n/a" if (b["market"] == "moneyline").any() else round(float(at_110) * 100, 1),
    })


def summarize_by(b, keys):
    rows = {name: summarize(group) for name, group in b.groupby(keys, observed=True)}
    return pd.DataFrame(rows).T.rename_axis(keys)


all_candidates = build_candidates(games[games["season"] >= FIRST_CALIBRATED_SEASON])
past_candidates = all_candidates[all_candidates["played"]]
hold_candidates = past_candidates[past_candidates["season"] >= HOLDOUT_START]

raw_bets = hold_candidates[hold_candidates["edge_raw"] >= RAW_MIN_EDGE]
cal_bets = hold_candidates[hold_candidates["edge"] >= CAL_MIN_EDGE]

print(f"HOLDOUT {HOLDOUT_START}–{int(hold_candidates['season'].max())}: {len(hold_candidates):,} possible bets\n")
print(f"RAW MODEL: bet when the raw edge is {RAW_MIN_EDGE:.0%} or more")
raw_table = summarize_by(raw_bets, "market"); raw_table.loc["ALL BETS"] = summarize(raw_bets); display(raw_table)
print(f"CALIBRATED: bet when the market-calibrated edge is {CAL_MIN_EDGE:.1%} or more")
if len(cal_bets):
    cal_table = summarize_by(cal_bets, "market"); cal_table.loc["ALL BETS"] = summarize(cal_bets); display(cal_table)
else:
    cal_table = pd.DataFrame()
    print("No bets qualified: after calibration, the model never disagreed with the closing line by enough to beat the vig.")

**How to read it**

- Compare `hit_rate_%` to `break_even_%`, never to 50%.
- If `roi_95%_range` includes zero, the result cannot be told apart from no edge.
- The calibrated strategy places far fewer bets. That is the calibration doing its job: it only speaks up when the model disagrees with the line by a lot.
- A small bet count means a wide uncertainty range. A few dozen bets prove nothing either way.

## 5d. Track record by confidence level

This is the table the website will show next to each pick. For every past game and market, take the side the raw model liked more, group by how big an edge it claimed, and see how those picks actually did. It uses every calibrated season (2010 onward) to get the largest honest sample.

In [ ]:
EDGE_BINS = [-1.0, 0.0, 0.03, 0.06, 0.10, 1.0]
EDGE_LABELS = ["No edge", "0–3%", "3–6%", "6–10%", "10%+"]

# One row per game and market: the side the raw model likes more.
record = past_candidates.sort_values("edge_raw", ascending=False).drop_duplicates(["game_id", "market"]).copy()
record["confidence"] = pd.cut(record["edge_raw"], bins=EDGE_BINS, labels=EDGE_LABELS)
track_record = summarize_by(record, ["market", "confidence"])
display(track_record)
track_record.reset_index().to_csv("nfl_track_record.csv", index=False)

If the model's confidence meant something, hit rate and ROI would rise from the top row of each market to the bottom. Read the ranges: with a few hundred bets per row, differences of two or three points of hit rate are within noise.

## 5e. Profit over time

In [ ]:
fig, ax = plt.subplots()
for label, b in (("Raw model, 3%+ edge", raw_bets), ("Calibrated, 0.5%+ edge", cal_bets)):
    if len(b):
        b = b.sort_values("gameday")
        ax.plot(b["gameday"], b["profit"].cumsum(), label=f"{label} ({len(b):,} bets)")
ax.axhline(0, color="k", linewidth=1)
ax.set_xlabel("Date"); ax.set_ylabel("Cumulative profit (units, 1 unit per bet)")
ax.set_title("Holdout backtest against closing lines")
ax.legend(loc="best", fontsize=9); plt.show()

by_season = raw_bets.groupby("season")["profit"].agg(bets="size", units="sum", roi="mean")
print(f"Raw model: winning seasons {(by_season['units'] > 0).sum()} of {len(by_season)}")
display(by_season.assign(roi=(by_season["roi"] * 100).round(1), units=by_season["units"].round(1)).rename(columns={"roi": "roi_%"}).T)

## 5f. The verdict, in plain English

This cell reads the tables above and states the result. It is written from the numbers each time the notebook runs, so it stays true as new games come in.

In [ ]:
m_base, m_raw, m_cal, m_mkt = scorecard["margin_rmse"].tolist()
w = info.loc["Spread: final model"]
print("VERDICT ON THE HOLDOUT SEASONS")
print()
print(f"1. Accuracy. The raw model misses the final margin by {m_raw:.2f} points on average; the closing spread misses by {m_mkt:.2f}.")
print(f"   The gap to the market is {m_raw - m_mkt:.2f} points, down from {m_base - m_mkt:.2f} for the baseline model.")
print()
proven_info = w["95%_low"] > 0
print(f"2. Information. About {w['weight_on_model']:.0%} of the model's disagreement with the spread shows up in results "
      f"(95% range {w['95%_low']:.0%} to {w['95%_high']:.0%}).")
print("   " + ("That is clearly above zero: the model knows something the closing line doesn't."
               if proven_info else "The range includes zero, so this is not proven."))
print()
print("3. Betting against closing lines.")
winners = []
for name, table, bets_df in (("Raw model", raw_table, raw_bets), ("Calibrated", cal_table, cal_bets)):
    for market in ("spread", "total", "moneyline"):
        if market not in table.index:
            continue
        r = table.loc[market]
        low, high = roi_range(bets_df[bets_df["market"] == market])
        at110 = "" if market == "moneyline" else f", {r['roi_at_-110_%']:+.1f}% at -110"
        print(f"   {name + ', ' + market + ':':24s}{r['hit_rate_%']:.1f}% hit rate (needed {r['break_even_%']:.1f}%) on {int(r['bets']):,} bets, "
              f"ROI {r['roi_%']:+.1f}%{at110}")
        if low > 0 and (market == "moneyline" or r["roi_at_-110_%"] > 0):
            winners.append(f"{name.lower()} {market}")
print()
if winners:
    print("BOTTOM LINE: " + ", ".join(winners) + " showed a profit whose 95% range stays above zero.")
    print("Treat that as a lead to confirm with live tracking, not as proof. Check for bugs first.")
else:
    print("BOTTOM LINE: no strategy shows a proven profit against NFL closing lines.")
    print("The model is a sound, honest pricing engine. It does not have a betting edge at the close.")

---
# 6. This week's games

Every upcoming game that already has a line, priced by the final model. Ratings include every game played so far this season.

For each game and market, the table shows the side the model prefers:

- **market_prob**: the vig-free probability implied by the current prices
- **model_prob**: the market-calibrated probability. This is the headline number.
- **raw_prob**: what the model would say with no calibration
- **edge**: `model_prob` minus the break-even probability at the listed price. Positive means the bet has positive expected value *if the model is right*.
- **track record**: how past picks at this confidence level did in the backtest, next to the hit rate they needed
- **model_lean**: `LEAN` when the calibrated edge clears the rule from section 5c

`LEAN` is not a recommendation to bet. It marks where the model sees value, and section 5 tells you whether that rule has actually made money. Show both on the website.

Two more cautions. These lines are whatever nflverse has recorded and may be stale, so check current prices. And upcoming games have no weather yet.

In [ ]:
upcoming = all_candidates[~all_candidates["played"]].copy()
if len(upcoming) == 0:
    print("No upcoming games with lines right now. Re-run this notebook when next week's lines are posted.")
    picks = pd.DataFrame()
else:
    # Keep the model's preferred side of each market in each game.
    upcoming = upcoming.sort_values("edge_raw", ascending=False).drop_duplicates(["game_id", "market"])
    upcoming["confidence"] = pd.cut(upcoming["edge_raw"], bins=EDGE_BINS, labels=EDGE_LABELS)
    lookup = track_record.reset_index()
    lookup["track_record"] = (lookup["hit_rate_%"].astype(str) + "% of " + lookup["bets"].astype(str)
                              + " (needs " + lookup["break_even_%"].astype(str) + "%)")
    upcoming = upcoming.merge(lookup[["market", "confidence", "track_record"]], on=["market", "confidence"], how="left")
    upcoming["model_lean"] = np.where(upcoming["edge"] >= CAL_MIN_EDGE, "LEAN", "-")
    picks = upcoming.sort_values(["gameday", "game_id", "market"])[[
        "gameday", "matchup", "market", "pick", "price", "market_prob", "model_prob", "model_prob_raw",
        "edge", "ev", "confidence", "track_record", "model_lean"]].rename(columns={"model_prob_raw": "raw_prob"})
    picks.to_csv("nfl_upcoming_picks.csv", index=False)

    show = picks.copy()
    show["gameday"] = show["gameday"].dt.strftime("%a %b %d")
    for col in ("market_prob", "model_prob", "raw_prob", "edge", "ev"):
        show[col] = (show[col] * 100).round(1)
    show["price"] = show["price"].map("{:+.0f}".format)
    print(f"{picks['matchup'].nunique()} upcoming games. Probabilities, edge and EV are in percent.")
    display(show.set_index(["gameday", "matchup", "market"]))
    n_leans = int((picks["model_lean"] == "LEAN").sum())
    print(f"\nPicks that clear the calibrated rule (edge of {CAL_MIN_EDGE:.1%} or more): {n_leans} of {len(picks)}")
    print("LEAN marks where the model sees value. Check section 5 before treating any of them as a bet.")

In [ ]:
# The model's own lines for each upcoming game, next to the market's.
next_games = games[~games["played"]].copy()
if len(next_games):
    board = pd.DataFrame({
        "date": next_games["gameday"].dt.strftime("%a %b %d"),
        "matchup": next_games["away_team"] + " at " + next_games["home_team"],
        "home_qb": next_games["home_qb_used"].map(qb_names).fillna("unknown"),
        "away_qb": next_games["away_qb_used"].map(qb_names).fillna("unknown"),
        "market_spread": -next_games["spread_line"],
        "raw_model_spread": -next_games["pred_margin"].round(1),
        "final_spread": -next_games["final_margin"].round(1),
        "market_total": next_games["total_line"],
        "raw_model_total": next_games["pred_total"].round(1),
        "final_total": next_games["final_total"].round(1),
        "home_win_prob_%": (next_games["p_win"] * 100).round(1),
    }).set_index(["date", "matchup"])
    print("Spreads are from the home team's side: -3 means the home team is favored by 3.")
    display(board)

---
# 7. Save everything the website needs

These files are the hand-off to the backend. The website never needs to retrain anything: it loads the model, the ratings and the track record.

| File | Contents |
|---|---|
| `nfl_model.joblib` | The trained margin and total models, calibration settings and input lists |
| `nfl_team_ratings.csv` | Current Elo, efficiency and quarterback ratings for all 32 teams |
| `nfl_upcoming_picks.csv` | This week's picks table |
| `nfl_track_record.csv` | Hit rate and ROI by market and confidence level |
| `nfl_backtest_predictions.csv` | Every out-of-sample prediction since 2010, for the site's performance page |

In [ ]:
def final_model(choice, feature_cols, target):
    # Refit the chosen model type on every finished game so it is as current as possible.
    train = games[games["played"] & (games["season"] >= FIRST_TRAIN_SEASON)]
    fitted = {}
    if choice in ("Baseline",):
        fitted["ridge"] = ridge().fit(train[MARGIN_BASE if target == "result" else TOTAL_BASE], train[target])
    if choice in ("Full ridge", "Blend"):
        fitted["ridge"] = ridge().fit(train[feature_cols], train[target])
    if choice in ("Boosted trees", "Blend"):
        fitted["boosted"] = boosted().fit(train[feature_cols], train[target])
    return fitted


latest_rows = games[games["season"] == CURRENT_SEASON].iloc[-1]
bundle = {
    "created": pd.Timestamp.now().isoformat(timespec="seconds"),
    "trained_through": str(games.loc[games["played"], "gameday"].max().date()),
    "margin": {"choice": MARGIN_CHOICE, "features": MARGIN_BASE if MARGIN_CHOICE == "Baseline" else MARGIN_FULL,
               "models": final_model(MARGIN_CHOICE, MARGIN_FULL, "result"),
               "c": float(latest_rows["c_margin"]), "sigma": float(latest_rows["sigma_margin"]), "sigma_raw": float(latest_rows["sigma_margin_raw"])},
    "total": {"choice": TOTAL_CHOICE, "features": TOTAL_BASE if TOTAL_CHOICE == "Baseline" else TOTAL_FULL,
              "models": final_model(TOTAL_CHOICE, TOTAL_FULL, "total"),
              "c": float(latest_rows["c_total"]), "sigma": float(latest_rows["sigma_total"]), "sigma_raw": float(latest_rows["sigma_total_raw"])},
    "rules": {"raw_min_edge": RAW_MIN_EDGE, "calibrated_min_edge": CAL_MIN_EDGE},
    "edge_bins": EDGE_BINS, "edge_labels": EDGE_LABELS,
}
joblib.dump(bundle, "nfl_model.joblib")

export = past_candidates[["game_id", "season", "gameday", "matchup", "market", "side", "pick", "price", "market_prob",
                          "model_prob", "model_prob_raw", "break_even", "edge", "edge_raw", "ev", "won", "profit"]]
export.to_csv("nfl_backtest_predictions.csv", index=False)

for name in ("nfl_model.joblib", "nfl_team_ratings.csv", "nfl_upcoming_picks.csv", "nfl_track_record.csv", "nfl_backtest_predictions.csv"):
    if os.path.exists(name):
        print(f"  saved {name:32s} {os.path.getsize(name) / 1024:8.0f} KB")
print(f"\nModel trained through {bundle['trained_through']}. Download the files from the folder icon on the left.")

---
# What the finished model is, and isn't

**What it is:** a complete, leak-free NFL pricing model. It rates every team and quarterback from play-by-play data, predicts margin and total, converts them to probabilities for moneyline, spread and total, calibrates those against the market, and reports each pick with an honest track record.

**How to judge it:** by the holdout tables in section 5, in this order.

1. *Forecast accuracy (5a):* how close is the final model to the closing line?
2. *Information test (5b):* is the weight on the model clearly above zero?
3. *Backtest (5c):* is `roi_at_-110_%` positive with a range that stays above zero?

If the answer to 3 is no, the model has **no proven betting edge against NFL closing lines**. That is the expected result: this is the most efficient betting market in sports, and the model does not know about injuries beyond the quarterback, weather forecasts, or late news. A model that matches the market's accuracy closely and is honestly calibrated is still a strong result and the right foundation.

**Where a real edge can still come from**

- **Earlier prices.** Closing lines are the sharpest. Lines posted early in the week are softer. Logging live odds and measuring whether the line moves toward the model's side (closing line value) is the real test, and it needs an odds feed.
- **Softer markets.** The same pipeline on college basketball or college football faces lines that get far less attention.
- **Line shopping.** The backtest shows how much the price matters: a few cents of vig is the difference between a small loss and break-even.

**What not to do:** change the settings at the top of section 2 until the holdout tables look better. The holdout only stays honest if it is looked at once.